🏛️ Mock Trial AI - Research & Development
This notebook contains the experimental data and the logic for the Phi-1.5 version of the legal assistant.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate
!pip install -q transformers trl peft datasets gradio huggingface_hub torch PyMuPDF

print("All libraries installed!")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import json
data = [
    # ---- WITNESS SIMULATOR EXAMPLES ----
    {
        "instruction": """You are James Carter testifying in People v. Davis.
Answer ONLY based on the case packet below. Do NOT invent facts.

CASE PACKET:
James Carter is a forensic analyst. On June 3rd Carter examined the
defendant's car and found traces of mud matching the crime scene location.
Carter also found a fiber on the front seat matching the victim's jacket.
Carter has 15 years experience in forensic analysis.""",
        "input": "What did you find when you examined the defendant's car?",
        "output": "I found two things. First, traces of mud on the undercarriage that matched the soil composition from the crime scene location. Second, a fiber on the front seat that matched the victim's jacket material."
    },
    {
        "instruction": """You are Sarah Mills testifying in People v. Davis.
Answer ONLY based on the case packet below. Do NOT invent facts.

CASE PACKET:
Sarah Mills is a neighbor who witnessed the incident. On June 3rd at
9pm Mills saw the defendant's blue Honda parked outside the victim's
house. Mills heard shouting coming from inside. Mills called 911 at
9:15pm. Mills did not see anyone enter or exit the house.""",
        "input": "What did you see on the night of June 3rd?",
        "output": "I saw the defendant's blue Honda parked outside the victim's house at around 9pm. I heard shouting coming from inside the house. I called 911 at 9:15pm. I did not see anyone enter or exit the house that night."
    },
    {
        "instruction": """You are Brian Todd testifying in People v. Davis.
Answer ONLY based on the case packet below. Do NOT invent facts.

CASE PACKET:
Brian Todd is the defendant's alibi witness. Todd states he was with
the defendant at a restaurant from 8pm to 10pm on June 3rd. Todd has
a receipt from the restaurant timestamped 9:32pm. Todd and the defendant
have been friends for 10 years.""",
        "input": "Where were you on the night of June 3rd?",
        "output": "I was with the defendant at Carmine's Restaurant from 8pm to 10pm. I have a receipt timestamped 9:32pm that night. We were together the entire time."
    },
    {
        "instruction": """You are a witness testifying in a mock trial.
Answer ONLY based on the case packet below. Do NOT invent facts.

CASE PACKET:
Dr. Anne Pierce is a medical examiner. Pierce determined the cause of
death was blunt force trauma to the head. Time of death was estimated
between 8:30pm and 10pm on June 3rd. Pierce found no defensive wounds
on the victim suggesting the attack was sudden.""",
        "input": "What was the cause of death?",
        "output": "The cause of death was blunt force trauma to the head. Based on my examination I estimated time of death between 8:30pm and 10pm. I found no defensive wounds which suggests the attack was sudden and unexpected."
    },
    {
        "instruction": """You are a witness testifying in a mock trial.
Answer ONLY based on the case packet below. Do NOT invent facts.

CASE PACKET:
Officer Ray Gomez responded to the 911 call at 9:22pm. Gomez found
the front door unlocked. Gomez discovered the victim unconscious in
the living room. Gomez secured the scene and called for backup.
Gomez did not touch or move anything before detectives arrived.""",
        "input": "What did you find when you arrived at the scene?",
        "output": "I arrived at 9:22pm in response to a 911 call. The front door was unlocked. I found the victim unconscious in the living room. I secured the scene and called for backup without touching or moving anything."
    },
    {
        "instruction": """You are a witness testifying in a mock trial.
Answer ONLY based on the case packet below. Do NOT invent facts.

CASE PACKET:
The witness has no information about the defendant's whereabouts.
The witness only observed the crime scene after the fact.""",
        "input": "Did you see the defendant at the scene?",
        "output": "I don't have information about that in my statement. I can only testify to what is in my witness statement and I have no information about the defendant's whereabouts."
    },

    # ---- OBJECTION EXAMPLES ----
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Prosecutor asks on direct examination: 'You saw the defendant running away, didn't you?'",
        "output": "OBJECTION — Leading Question. On direct examination the attorney cannot suggest the answer to their own witness. Proper form: 'What if anything did you see the defendant doing after the incident?'"
    },
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Defense asks: 'What is your name, how long have you been a detective, and what did you find at the scene?'",
        "output": "OBJECTION — Compound Question. This asks three separate questions at once. Each should be asked individually. Break it into: 'What is your name?' then 'How long have you been a detective?' then 'What did you find at the scene?'"
    },
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Defense asks the expert witness: 'Do you think the defendant intended to commit this crime?'",
        "output": "OBJECTION — Improper Opinion on Ultimate Issue. An expert cannot opine on whether the defendant had a specific mental state as that is the ultimate question for the jury. The expert may testify about facts and observations but not the defendant's intent."
    },
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Prosecutor asks: 'Tell the jury everything you know about what happened that night.'",
        "output": "OBJECTION — Calls for a Narrative. This question is too broad and unguided. The attorney must ask specific focused questions rather than inviting the witness to tell an open ended story."
    },
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Defense asks on cross examination: 'Isn't it true you have been wrong before in your forensic analysis?'",
        "output": "NO OBJECTION warranted. This is a proper cross examination question challenging the witness's credibility and past accuracy. Cross examination allows leading questions and credibility challenges."
    },
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Prosecutor asks the detective: 'What did the witness tell you she saw that night?'",
        "output": "OBJECTION — Hearsay. The detective is being asked to repeat an out of court statement to prove the truth of what the witness saw. The witness should testify directly rather than having the detective relay their statement."
    },
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Defense asks: 'You have no idea where my client was that night, do you?'",
        "output": "OBJECTION — Argumentative. This question is accusatory and designed to argue rather than elicit facts. Proper form: 'Do you have any personal knowledge of the defendant's whereabouts that night?'"
    },
    {
        "instruction": "You are a mock trial objection trainer. Identify if this question warrants an objection and explain why.",
        "input": "Prosecutor asks the forensic expert: 'Is DNA analysis a reliable scientific method?'",
        "output": "NO OBJECTION warranted. This is a proper foundational question for an expert witness establishing the reliability of the scientific method they used. It is within the scope of their expertise."
    },

    # ---- CASE ANALYSIS EXAMPLES ----
    {
        "instruction": """You are a legal analyst. Analyze the case packet below and answer the question.

CASE PACKET:
People v. Davis. Defendant accused of assault.
Prosecution evidence: forensic mud match, fiber evidence, neighbor heard shouting.
Defense evidence: restaurant receipt alibi timestamped 9:32pm, alibi witness.
Victim found unconscious at 9:22pm. Time of death window 8:30-10pm.""",
        "input": "What is the prosecution's strongest evidence?",
        "output": "The prosecution's strongest evidence is: (1) Forensic mud on the defendant's car matching the crime scene. (2) A fiber on the front seat matching the victim's jacket. (3) A neighbor who heard shouting from the victim's house at 9pm. Together these place the defendant at the scene physically."
    },
    {
        "instruction": """You are a legal analyst. Analyze the case packet below and answer the question.

CASE PACKET:
People v. Davis. Defendant accused of assault.
Prosecution evidence: forensic mud match, fiber evidence, neighbor heard shouting.
Defense evidence: restaurant receipt alibi timestamped 9:32pm, alibi witness.
Victim found unconscious at 9:22pm. Time of death window 8:30-10pm.""",
        "input": "What is the defense's strongest argument?",
        "output": "The defense's strongest argument is the alibi. A restaurant receipt timestamped 9:32pm places the defendant elsewhere during the time of the attack. Combined with a corroborating alibi witness this creates significant reasonable doubt about the defendant's presence at the scene."
    },
    {
        "instruction": """You are a legal analyst. Analyze the case packet below and answer the question.

CASE PACKET:
People v. Davis. Defendant accused of assault.
Key witnesses: forensic analyst Carter, neighbor Mills, alibi witness Todd, medical examiner Pierce.
Physical evidence: mud sample, fiber, restaurant receipt.""",
        "input": "Who are the key witnesses and what does each contribute?",
        "output": "The key witnesses are: (1) Carter — forensic analyst who ties the defendant's car to the crime scene physically. (2) Mills — neighbor who places the defendant's car at the scene and heard shouting. (3) Todd — alibi witness who supports the defense timeline. (4) Pierce — medical examiner who establishes time of death and manner of injury."
    },
    # Detective Rio Sacks Statement
    {
       "instruction": """You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.

       Detective Rio Sacks' Witness Statement:
       In the cubby area, I located a jacket in a cubby that Rodriguez
informed me belonged to Haley Fromholz. Rodriguez explained
that Fromholz had entered the set the morning of April 9 wearing
that jacket and had removed it and placed it in their individual
cubby just prior to roll call. Rodriguez explained that minutes before
filming, Haley Fromholz had accessed this jacket. I also learned of
the existence of a green plastic baggie. While off-camera, in the
cubby area, Rodriguez witnessed Fromholz grab Fromholz’s jacket
and pull from the jacket pocket a green plastic baggie. Rodriguez
did not see what was taken out of the baggie but did notice Haley
Fromholz walking back to Fromholz’s station with a closed fist, as if
Fromholz was holding something. I took note of this information
and secured the jacket as evidence to send into the lab for fiber
analysis. On the morning of April 12, the medical examiner completed
Sears’s autopsy, confirming that his cause of death was attributed
to heartstopper mushrooms. Though the medical examiner noted
that Sears suffered from fatty liver disease, which contributed to
the unusually accelerated timeline of poisoning onset, symptoms,
and death; it was the heartstopper mushrooms that killed him. That
afternoon, I received a report from the lab on Fromholz’s jacket that
I had sent in for fiber analysis. I learned from our forensic analyst
fiber specialists that heartstopper mushroom spores were detected
on Fromholz’s jacket sleeve, along the outer cuff of the jacket’s left
sleeve. Next-generation sequencing, a powerful technology that
© 2025, Teach Democracy 39
People v. Fromholz
1
2
3
4
5
6
7
8
9
10
11
12
helps sequence DNA directly from a sample, was used to
determine the DNA of the mushroom and to determine that the
spores found on Fromholz’s jacket were of the heartstopper
species. The number of spores found on Fromholz’s jacket showed
that Fromholz had not merely brushed by the mushrooms but had
actively handled them (having contact with the gills where spores
are released)""",
"input": "From a prosecution view, how does the fiber evidence link Fromholz to Searss death?",
"output": "The fiber evidence strongly links Fromholz to Sears's death because heartstopper mushroom spores—confirmed as the exact species via next-generation sequencing—were found on the outer cuff of Fromholz's left jacket sleeve. The high spore count indicates active handling (contact with gills), not casual brushing, tying back to Rodriguez witnessing Fromholz pull a green baggie from that jacket pocket just before filming."
    },
    {
         "instruction": """You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.

       Detective Rio Sacks' Witness Statement:
       In the cubby area, I located a jacket in a cubby that Rodriguez
informed me belonged to Haley Fromholz. Rodriguez explained
that Fromholz had entered the set the morning of April 9 wearing
that jacket and had removed it and placed it in their individual
cubby just prior to roll call. Rodriguez explained that minutes before
filming, Haley Fromholz had accessed this jacket. I also learned of
the existence of a green plastic baggie. While off-camera, in the
cubby area, Rodriguez witnessed Fromholz grab Fromholz’s jacket
and pull from the jacket pocket a green plastic baggie. Rodriguez
did not see what was taken out of the baggie but did notice Haley
Fromholz walking back to Fromholz’s station with a closed fist, as if
Fromholz was holding something. I took note of this information
and secured the jacket as evidence to send into the lab for fiber
analysis. On the morning of April 12, the medical examiner completed
Sears’s autopsy, confirming that his cause of death was attributed
to heartstopper mushrooms. Though the medical examiner noted
that Sears suffered from fatty liver disease, which contributed to
the unusually accelerated timeline of poisoning onset, symptoms,
and death; it was the heartstopper mushrooms that killed him. That
afternoon, I received a report from the lab on Fromholz’s jacket that
I had sent in for fiber analysis. I learned from our forensic analyst
fiber specialists that heartstopper mushroom spores were detected
on Fromholz’s jacket sleeve, along the outer cuff of the jacket’s left
sleeve. Next-generation sequencing, a powerful technology that
© 2025, Teach Democracy 39
People v. Fromholz
1
2
3
4
5
6
7
8
9
10
11
12
helps sequence DNA directly from a sample, was used to
determine the DNA of the mushroom and to determine that the
spores found on Fromholz’s jacket were of the heartstopper
species. The number of spores found on Fromholz’s jacket showed
that Fromholz had not merely brushed by the mushrooms but had
actively handled them (having contact with the gills where spores
are released)""",
"input": "From a defense view, what doubts could you raise about the baggie incident or spore transfer?",
"output": "Defense can argue reasonable doubt by noting Rodriguez didn't see what (if anything) Fromholz removed from the baggie or placed in her closed fist, making the handling speculative. Spores could transfer innocently in a shared cubby area on a cooking set with foraging themes, and no direct link exists to the mushrooms in Sears's tlayuda—especially since fatty liver disease accelerated the death timeline unusually."
    },
    {
         "instruction": """You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.

       Detective Rio Sacks' Witness Statement:
       In the cubby area, I located a jacket in a cubby that Rodriguez
informed me belonged to Haley Fromholz. Rodriguez explained
that Fromholz had entered the set the morning of April 9 wearing
that jacket and had removed it and placed it in their individual
cubby just prior to roll call. Rodriguez explained that minutes before
filming, Haley Fromholz had accessed this jacket. I also learned of
the existence of a green plastic baggie. While off-camera, in the
cubby area, Rodriguez witnessed Fromholz grab Fromholz’s jacket
and pull from the jacket pocket a green plastic baggie. Rodriguez
did not see what was taken out of the baggie but did notice Haley
Fromholz walking back to Fromholz’s station with a closed fist, as if
Fromholz was holding something. I took note of this information
and secured the jacket as evidence to send into the lab for fiber
analysis. On the morning of April 12, the medical examiner completed
Sears’s autopsy, confirming that his cause of death was attributed
to heartstopper mushrooms. Though the medical examiner noted
that Sears suffered from fatty liver disease, which contributed to
the unusually accelerated timeline of poisoning onset, symptoms,
and death; it was the heartstopper mushrooms that killed him. That
afternoon, I received a report from the lab on Fromholz’s jacket that
I had sent in for fiber analysis. I learned from our forensic analyst
fiber specialists that heartstopper mushroom spores were detected
on Fromholz’s jacket sleeve, along the outer cuff of the jacket’s left
sleeve. Next-generation sequencing, a powerful technology that
© 2025, Teach Democracy 39
People v. Fromholz
1
2
3
4
5
6
7
8
9
10
11
12
helps sequence DNA directly from a sample, was used to
determine the DNA of the mushroom and to determine that the
spores found on Fromholz’s jacket were of the heartstopper
species. The number of spores found on Fromholz’s jacket showed
that Fromholz had not merely brushed by the mushrooms but had
actively handled them (having contact with the gills where spores
are released)""",
"input": "What was confirmed as Sears's cause of death, and what contributing factor was noted in the autopsy?",
"output": "Sears's autopsy confirmed heartstopper mushrooms as the cause of death. Fatty liver disease was noted as a contributing factor, accelerating the poisoning's onset, symptoms, and death timeline."
    },
    #Sam Longo's Statement
    {
        "instruction": """You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.

        Sam Longo's Witness Statement:
        The defendant Haley Fromholz had used Google Chromes private
browsing, or “incognito” mode, so that none of their history cookies or
search records were saved. But even in incognito mode, the laptop still
communicates with the internet. It saves pieces of information like
web addresses in hidden areas called a “cache.” I was able to create a
digital clone of the laptops storage so I could search deep system files
without changing anything on the primary laptop. I used specialized
software to search every corner of the copy, even temporary files and
deleted data. The individual data points are known as artifacts and
can be analyzed by the recovery software.
I applied a keyword search for terms related to the case such as
“poison mushrooms in Oakview,” “heartstopper mushrooms,”
“heartstopper poisoning,” and “Artemis Grove.” Even though Haley
Fromholz had cleared their history and used incognito mode, the
laptop still left behind traces and these traces matched the keywords.
Once I was able to recover the deleted searches, I could see that the
laptop had been used to visit web pages listing sightings of
heartstopper mushrooms and other toxic mushrooms found in the Bay
Area on April 8 between 4:00 p.m. and 5:30 p.m. The most frequent
website visited was the citizen-science website iForage. Fromholz
accessed one post on this page several times over the timespan
between 4:00 p.m. and 5:30 p.m., a page titled “Heartstopper
mushroom in Artemis Grove, Oakview on March 29, 2025,” It shows a
photograph of the mushroom at the trunk of an oak tree and has clear
directions on where to find it in the southern end of the park, as well
as the GPS coordinates for its location in the park. In the comment
section of the post, the user who uploaded the photo and coordinates
had written: “Easily accessible from the main trail leading toward the
southern end of Artemis from the northern parking lot, just look for the
huge oak next to the trail sign, on the Old Oak Loop.” These were the
only searches I found on the laptop that were relevant to this criminal
investigation according to the information I was provided by Detective
Sacks.
I did see evidence that Haley Fromholz semi-routinely wipes the
search history on their laptop, but there was no clear pattern here. I
had access to Fromholz’s laptop search data from twelve months prior
to the present, enabling me to look at certain search patterns or
behaviors. I could see that Fromholz’s search history had been cleared
six times over the past year: once on May 5 of last year, then once on
July 11, once on November 22, once on January 9 and again on
January 29. The most recent occurrence was April 8. On April 8,
Fromholz cleared all recent search history on their laptop at 5:35 p.m.,
minutes after the last search related to the investigation had been
completed.""",
"input": 'From the prosecutions perspective, how does this evidence suggest premeditation?',
"output": "Incognito mode and immediate post-search wipe (5:35 p.m. April 8) after repeated views of exact heartstopper location (iForage post with directions/GPS) show intent to research, locate, and hide sourcing poisonous mushrooms tied to the case keywords."
    },
    {
    "instruction": """You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.

        Sam Longo's Witness Statement:
        The defendant Haley Fromholz had used Google Chromes private
browsing, or “incognito” mode, so that none of their history cookies or
search records were saved. But even in incognito mode, the laptop still
communicates with the internet. It saves pieces of information like
web addresses in hidden areas called a “cache.” I was able to create a
digital clone of the laptops storage so I could search deep system files
without changing anything on the primary laptop. I used specialized
software to search every corner of the copy, even temporary files and
deleted data. The individual data points are known as artifacts and
can be analyzed by the recovery software.
I applied a keyword search for terms related to the case such as
“poison mushrooms in Oakview,” “heartstopper mushrooms,”
“heartstopper poisoning,” and “Artemis Grove.” Even though Haley
Fromholz had cleared their history and used incognito mode, the
laptop still left behind traces and these traces matched the keywords.
Once I was able to recover the deleted searches, I could see that the
laptop had been used to visit web pages listing sightings of
heartstopper mushrooms and other toxic mushrooms found in the Bay
Area on April 8 between 4:00 p.m. and 5:30 p.m. The most frequent
website visited was the citizen-science website iForage. Fromholz
accessed one post on this page several times over the timespan
between 4:00 p.m. and 5:30 p.m., a page titled “Heartstopper
mushroom in Artemis Grove, Oakview on March 29, 2025,” It shows a
photograph of the mushroom at the trunk of an oak tree and has clear
directions on where to find it in the southern end of the park, as well
as the GPS coordinates for its location in the park. In the comment
section of the post, the user who uploaded the photo and coordinates
had written: “Easily accessible from the main trail leading toward the
southern end of Artemis from the northern parking lot, just look for the
huge oak next to the trail sign, on the Old Oak Loop.” These were the
only searches I found on the laptop that were relevant to this criminal
investigation according to the information I was provided by Detective
Sacks.
I did see evidence that Haley Fromholz semi-routinely wipes the
search history on their laptop, but there was no clear pattern here. I
had access to Fromholz’s laptop search data from twelve months prior
to the present, enabling me to look at certain search patterns or
behaviors. I could see that Fromholz’s search history had been cleared
six times over the past year: once on May 5 of last year, then once on
July 11, once on November 22, once on January 9 and again on
January 29. The most recent occurrence was April 8. On April 8,
Fromholz cleared all recent search history on their laptop at 5:35 p.m.,
minutes after the last search related to the investigation had been
completed.""",
"input": "From the defense's perspective, how is this evidence weak or challengeable?",
"output": "No pattern to wiping (semi-routine, 6x/year), so April 8 isn't suspicious; searches could be innocent curiosity (Bay Area sightings common); cache artifacts recoverable from anyone using public/incognito—lacks proof Fromholz (vs. shared laptop) did them or acted on info; only relevant per Sacks info, potentially biased scope."
    },
    {
          "instruction": """You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.

        Sam Longo's Witness Statement:
        The defendant Haley Fromholz had used Google Chromes private
browsing, or “incognito” mode, so that none of their history cookies or
search records were saved. But even in incognito mode, the laptop still
communicates with the internet. It saves pieces of information like
web addresses in hidden areas called a “cache.” I was able to create a
digital clone of the laptops storage so I could search deep system files
without changing anything on the primary laptop. I used specialized
software to search every corner of the copy, even temporary files and
deleted data. The individual data points are known as artifacts and
can be analyzed by the recovery software.
I applied a keyword search for terms related to the case such as
“poison mushrooms in Oakview,” “heartstopper mushrooms,”
“heartstopper poisoning,” and “Artemis Grove.” Even though Haley
Fromholz had cleared their history and used incognito mode, the
laptop still left behind traces and these traces matched the keywords.
Once I was able to recover the deleted searches, I could see that the
laptop had been used to visit web pages listing sightings of
heartstopper mushrooms and other toxic mushrooms found in the Bay
Area on April 8 between 4:00 p.m. and 5:30 p.m. The most frequent
website visited was the citizen-science website iForage. Fromholz
accessed one post on this page several times over the timespan
between 4:00 p.m. and 5:30 p.m., a page titled “Heartstopper
mushroom in Artemis Grove, Oakview on March 29, 2025,” It shows a
photograph of the mushroom at the trunk of an oak tree and has clear
directions on where to find it in the southern end of the park, as well
as the GPS coordinates for its location in the park. In the comment
section of the post, the user who uploaded the photo and coordinates
had written: “Easily accessible from the main trail leading toward the
southern end of Artemis from the northern parking lot, just look for the
huge oak next to the trail sign, on the Old Oak Loop.” These were the
only searches I found on the laptop that were relevant to this criminal
investigation according to the information I was provided by Detective
Sacks.
I did see evidence that Haley Fromholz semi-routinely wipes the
search history on their laptop, but there was no clear pattern here. I
had access to Fromholz’s laptop search data from twelve months prior
to the present, enabling me to look at certain search patterns or
behaviors. I could see that Fromholz’s search history had been cleared
six times over the past year: once on May 5 of last year, then once on
July 11, once on November 22, once on January 9 and again on
January 29. The most recent occurrence was April 8. On April 8,
Fromholz cleared all recent search history on their laptop at 5:35 p.m.,
minutes after the last search related to the investigation had been
completed.""",
"input": "Describe the full process the analyst used to examine Fromholz's laptop without altering the original.",
"output": "The analyst created a digital clone of the laptop’s storage to avoid changing anything on the primary device. Specialized software was then used to search every corner of this copy, including deep system files, temporary files, and deleted data. Individual data points, known as artifacts, were analyzed by the recovery software."
    },
#Testing AI for more examples
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. On the evening of October 12, a convenience store in a suburban neighborhood was robbed shortly before closing. The store clerk, Daniel Ruiz, reported that a masked individual entered at approximately 9:52 PM, demanded money, and implied possession of a weapon, though none was visibly displayed. Ruiz stated that the suspect wore a dark hoodie, gloves, and spoke in a low voice, possibly to conceal identity. A customer, Lila Chen, who was outside the store during the incident, claimed she saw a person matching that description run out and enter a blue sedan parked nearby with its engine running. Surveillance footage partially corroborates this, showing a figure exiting the store and entering a vehicle, though the license plate is not clearly visible. Another witness, Marcus Bell, a neighbor across the street, reported hearing a car accelerate quickly and noted that he had seen a similar blue sedan circling the block roughly 15 minutes earlier. Police later identified a suspect, Aaron Pike, whose vehicle matches the description. However, Pike claims he was in the area visiting a friend and denies involvement. No weapon was recovered, and no fingerprints conclusively tied Pike to the scene, though fibers consistent with his clothing were found inside the store.",
  "input": "How does the witness testimony regarding the blue sedan affect the prosecution's case theory?",
  "output": "The witness testimony regarding the blue sedan strengthens the prosecution's case by linking the suspect to both the scene and the timing of the crime. Lila Chen's observation of a person exiting the store and entering a blue sedan provides a direct connection between the fleeing suspect and a specific vehicle. Marcus Bell's statement that a similar vehicle had been circling the area shortly before the robbery suggests premeditation and surveillance, reinforcing intent. When combined with the fact that Aaron Pike owns a vehicle matching that description, the prosecution can argue that this establishes circumstantial evidence placing Pike at the scene during the crime and potentially planning the robbery in advance."
    },
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A late-night altercation occurred outside a downtown restaurant resulting in serious injury to one individual, Kevin Morales. Witnesses report that Morales and another man, identified as Jordan Lee, were engaged in a heated verbal dispute that escalated into physical violence. A server, Ana Patel, stated that she saw Lee push Morales first, after which Morales stumbled but attempted to regain balance. Another witness, Trevor Kim, claimed that Morales swung first, striking Lee in the shoulder before Lee retaliated. Security footage captures part of the incident but is partially obstructed by parked cars, making it unclear who initiated physical contact. Lee claims self-defense, stating that Morales threatened him verbally and made aggressive gestures before any physical interaction occurred. Morales, now hospitalized, has no clear recollection of the sequence of events due to a head injury sustained during the incident. No weapons were involved, but Morales suffered a severe concussion after falling and hitting the pavement.",
  "input": "How does Trevor Kim's testimony benefit the defense's case theory?",
  "output": "Trevor Kim's testimony supports the defense's claim of self-defense by asserting that Kevin Morales initiated the physical confrontation. If Morales struck first, this provides a legal basis for Jordan Lee to respond with reasonable force to protect himself. This testimony introduces doubt about the prosecution's narrative that Lee was the aggressor and aligns with Lee's own account of events. Even though the video footage is unclear, Kim's statement helps establish an alternative sequence in which Lee's actions were reactive rather than initiated, which is critical for a self-defense argument."
    },
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A warehouse fire broke out in an industrial district at approximately 2:15 AM, causing extensive property damage but no injuries. Fire investigators determined that the fire originated near a storage area containing flammable chemicals. The warehouse owner, Samuel Ortiz, reported that all safety protocols were followed and that no unauthorized personnel should have been present. However, a security guard, Elena Vasquez, stated that she saw an individual inside the warehouse after hours, approximately 30 minutes before the fire began. She described the person as wearing a reflective vest and carrying what appeared to be a toolbox. Another employee, Ryan Cole, mentioned that a contractor had been scheduled to perform maintenance earlier that day but was not authorized to remain after closing. Surveillance cameras inside the warehouse were non-functional due to a reported system outage. Investigators later found evidence of accelerants inconsistent with normal warehouse materials, suggesting possible arson. The contractor in question denies returning to the warehouse after hours.",
  "input": "What key fact in this case packet suggests the possibility of arson rather than an accidental fire?",
  "output": "The discovery of accelerants that are inconsistent with the materials normally stored in the warehouse is the key fact suggesting arson. This indicates that a substance was intentionally introduced to accelerate the fire, which would not occur in an accidental fire scenario. Combined with the report of an unauthorized individual inside the warehouse shortly before the fire, this evidence supports the theory that the fire was deliberately set."
    },
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. On March 3rd, a residential burglary occurred in a quiet suburban neighborhood during the early afternoon. The homeowner, Priya Shah, reported returning home at approximately 2:40 PM to find her back door forced open and several valuables missing, including jewelry and a laptop. A neighbor, Thomas Greene, stated that he observed a young man walking up the driveway around 2:10 PM, wearing a baseball cap and carrying a backpack. Greene initially assumed the individual was a delivery worker but later found the behavior suspicious when the individual remained at the house for an extended period. Another neighbor, Alisha Wong, reported hearing a loud noise consistent with a door being kicked open but did not visually confirm the suspect. Surveillance footage from a nearby home shows a figure entering the property, but the face is obscured. Police later identified a suspect, Dylan Cruz, who was found in possession of a similar laptop but claims he purchased it from an unknown seller earlier that day. No direct forensic evidence places Cruz inside the home.",
  "input": "How does Thomas Greene's testimony benefit the prosecution's case theory?",
  "output": "Thomas Greene's testimony supports the prosecution by placing an unidentified individual at the victim's home during the time window of the burglary. His observation that the individual approached the house and remained there for an extended period suggests opportunity and potential involvement. This aligns with the timeline of the crime and reinforces the idea that the suspect was not merely passing by. When combined with the later discovery of stolen property in Dylan Cruz's possession, the testimony helps build a circumstantial case linking the suspect to the burglary."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A hit-and-run incident occurred at approximately 8:25 PM at a busy intersection. The victim, Laura Simmons, was struck while crossing the street and sustained multiple injuries. A witness, Eric Nolan, stated that he saw a dark-colored SUV run a red light and hit Simmons before speeding away. Another witness, Jasmine Patel, claimed the light may have been yellow and that the pedestrian may have stepped into the crosswalk late. Traffic camera footage is partially obscured due to heavy rain, making it difficult to determine the exact signal color at the time of impact. Investigators later identified a suspect vehicle with front-end damage matching the incident, owned by Victor Hale. Hale claims he was driving in the area but insists he did not hit anyone and suggests the damage came from a prior accident. No clear DNA or physical evidence ties Hale directly to the victim.",
  "input": "How does Jasmine Patel's testimony benefit the defense's case theory?",
  "output": "Jasmine Patel's testimony introduces uncertainty about whether the driver committed a clear traffic violation. By suggesting that the light may have been yellow and that the pedestrian may have entered the crosswalk late, her account supports the possibility that the incident was not entirely the driver's fault. This helps the defense argue that the situation was ambiguous and that liability cannot be definitively assigned. It also weakens the prosecution's claim that the driver acted recklessly or negligently."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. An alleged embezzlement case involves a mid-level accountant, Rachel Kim, working at a small tech firm. Over a period of six months, discrepancies totaling approximately $75,000 were discovered in company accounts. The firm's CFO, Daniel Brooks, stated that Kim had exclusive access to the affected accounts and was responsible for reconciling them. A coworker, Megan Liu, reported noticing that Kim frequently stayed late and appeared anxious when asked about financial reports. However, another employee, Brian Scott, stated that multiple team members, including Brooks himself, had access to the accounting software. Internal audit logs show irregular access times but do not conclusively identify who made the unauthorized transactions. Kim denies any wrongdoing and claims the discrepancies may be due to a software glitch or accounting error.",
  "input": "What fact in this case packet weakens the prosecution's claim that Rachel Kim acted alone?",
  "output": "The fact that multiple employees, including Daniel Brooks, had access to the accounting software weakens the prosecution's claim that Rachel Kim acted alone. This introduces the possibility that others could have made the unauthorized transactions, thereby creating reasonable doubt about exclusive responsibility. Without clear audit logs identifying the user responsible for each transaction, the prosecution cannot definitively attribute the discrepancies solely to Kim."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A physical altercation occurred at a college dormitory resulting in injuries to one student, Alex Romero. According to Romero, another student, Ethan Park, confronted him over a personal dispute and initiated physical contact by shoving him against a wall. A resident advisor, Kelly Tran, stated that she heard raised voices and arrived shortly after the incident began, observing both individuals engaged in a struggle. Another student, Noah Feldman, claimed that Romero had been acting aggressively earlier in the evening and may have provoked the confrontation. There is no video footage of the incident, and both parties provide conflicting accounts. Park claims he acted in self-defense after Romero approached him in a threatening manner. Medical reports confirm that Romero sustained a fractured wrist during the altercation.",
  "input": "How does Noah Feldman's testimony affect the defense's case theory?",
  "output": "Noah Feldman's testimony supports the defense by suggesting that Alex Romero may have been the initial aggressor. If Romero acted aggressively or provoked the confrontation, this strengthens Ethan Park's claim of self-defense. The testimony introduces doubt about the prosecution's narrative and provides an alternative explanation for why Park may have used force. This is especially important in the absence of clear video evidence."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A late-night robbery occurred at a gas station where the cashier, Miguel Torres, reported that a suspect entered the store at approximately 11:15 PM and demanded cash while keeping one hand inside his jacket, implying he had a weapon. Torres complied, and the suspect fled with approximately $600. A customer outside, Rebecca Lin, stated that she saw a man exit the store quickly and get into a red sedan parked nearby. Another witness, Omar Haddad, claimed that he saw two individuals inside the vehicle, suggesting a possible accomplice. Surveillance footage confirms a red sedan leaving the scene but does not clearly show the occupants. Police later stopped a vehicle matching the description and found a suspect, Luis Navarro, inside with a similar amount of cash. Navarro claims he was not involved and that the money came from his job earlier that evening. No weapon was found.",
  "input": "What key fact supports the theory that more than one person may have been involved in the robbery?",
  "output": "Omar Haddad's statement that he observed two individuals inside the red sedan supports the theory that more than one person was involved. This suggests the presence of an accomplice, potentially indicating coordination or planning. The existence of a getaway driver would align with common robbery patterns and strengthens the argument that the crime was not committed by a single individual acting alone."
},
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. On July 18th, a bicycle theft was reported outside a public library in a busy downtown area around 5:30 PM. The owner, Kevin Das, stated that he had locked his bicycle to a rack using a cable lock before entering the building for approximately 25 minutes. Upon returning, he found the lock cut and the bicycle missing. A nearby street vendor, Luis Ortega, reported seeing a person wearing a gray hoodie crouched near the bike rack during that time, appearing to manipulate something near a bicycle. Ortega stated he did not intervene because he assumed the individual was unlocking their own bike. Another witness, Dana Brooks, claimed she saw a person quickly riding away on a bicycle and dropping what appeared to be a pair of bolt cutters into a nearby trash can. Police later recovered bolt cutters from that location. A suspect, Marcus Hill, was later stopped nearby with a bicycle matching the description, but he claimed he had purchased it from someone on the street moments earlier and denied cutting any lock.",
  "input": "How does Dana Brooks' testimony support the prosecution's case theory?",
  "output": "Dana Brooks' testimony supports the prosecution by providing a direct observation of suspicious behavior consistent with theft. Her statement that the individual discarded bolt cutters suggests an attempt to dispose of incriminating evidence immediately after taking the bicycle. This aligns with the recovery of the tool and strengthens the inference that the person seen fleeing was responsible for cutting the lock. When combined with the suspect being found in possession of the stolen bicycle shortly afterward, it reinforces the prosecution's argument that the defendant committed the theft."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A dispute occurred at a local electronics store involving an alleged shoplifting incident. Store security personnel reported that a customer, Nina Verma, was seen placing a small electronic device into her bag without paying. The security guard, Paul Jenkins, stated that he observed this behavior through in-store cameras and detained Verma near the exit. Verma claims that she had intended to pay for the item and was still browsing when she was stopped. Another customer, Olivia Chen, stated that she saw Verma holding the item openly earlier and did not notice any attempt to conceal it. Store camera footage shows Verma holding the item and later placing it into her bag, but the angle does not clearly show her intent or whether she was proceeding toward checkout. No prior incidents involving Verma were reported.",
  "input": "How does Olivia Chen's testimony benefit the defense's case theory?",
  "output": "Olivia Chen's testimony benefits the defense by suggesting that Nina Verma did not act with clear intent to steal. Her observation that the item was previously held openly indicates that Verma was not attempting to conceal it from the outset. This supports the argument that placing the item in her bag may have been temporary or part of continued shopping rather than theft. The testimony introduces reasonable doubt about whether Verma had the necessary intent required for a shoplifting charge."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A vandalism incident occurred at a public park where several benches were spray-painted overnight. City maintenance staff discovered the damage early in the morning and reported it to authorities. A jogger, Steven Park, stated that he saw two individuals in dark clothing near the benches at approximately 11:45 PM the previous night, one of whom appeared to be holding a spray can. Another witness, Rachel Adams, reported hearing aerosol spraying sounds from a distance but did not see anyone. Surveillance cameras in the park were temporarily offline due to maintenance. Police later questioned a local teenager, Jason Cole, who had paint residue on his hands but claimed he had been working on an art project at home. No direct footage or fingerprints were recovered from the benches.",
  "input": "What fact in this case packet creates uncertainty about Jason Cole's involvement?",
  "output": "The lack of direct evidence linking Jason Cole to the vandalized benches creates uncertainty about his involvement. Although he had paint residue on his hands, there is no forensic proof, such as fingerprints or video footage, placing him at the park during the incident. Additionally, his explanation that he was working on an art project provides an alternative source for the paint. This absence of direct linkage weakens the ability to conclusively connect him to the crime."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A late-night noise complaint escalated into an alleged assault at an apartment complex. The complainant, Sarah Malik, stated that she went to her neighbor's unit to request that loud music be turned down and was met by an aggressive individual, Chris Dalton, who she claims shoved her during the interaction. Dalton denies the allegation and claims that Malik was the one who became confrontational and entered his personal space. Another resident, Leo Martinez, reported hearing raised voices but did not witness any physical contact. A second neighbor, Emily Ross, stated that she briefly saw the interaction and believed both individuals were arguing loudly but could not determine who initiated physical contact. No video footage exists, and no visible injuries were documented immediately after the incident.",
  "input": "How does the lack of visible injuries impact the defense's case theory?",
  "output": "The absence of visible injuries supports the defense by raising questions about whether a physical assault occurred as described. If no physical harm is evident, it may suggest that any contact was minimal or did not occur at all. This weakens the credibility of the claim that significant force was used. Combined with conflicting witness accounts, it helps the defense argue that the evidence is insufficient to prove an assault beyond a reasonable doubt."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A delivery package theft was reported in a residential neighborhood where multiple packages had gone missing over several days. One homeowner, Linda Cho, installed a doorbell camera that captured a person approaching her porch, picking up a package, and leaving quickly. The individual wore sunglasses and a hat, obscuring identifying features. A neighbor, Greg Fuller, stated that he saw a similar individual walking down the street earlier carrying multiple packages. Another resident, Priyank Sethi, claimed that package thefts had been occurring frequently and that multiple suspects could be involved. Police later stopped a suspect, Evan Ross, who was found with several unopened packages in his vehicle. Ross claims he was collecting packages for a friend who had recently moved and denies any theft. No fingerprints or direct identification link Ross to the specific incident captured on camera.",
  "input": "How does Priyank Sethi's statement affect the prosecution's case theory?",
  "output": "Priyank Sethi's statement introduces the possibility that multiple individuals may be responsible for the package thefts, which weakens the prosecution's effort to attribute all incidents to Evan Ross. If multiple suspects are involved, it becomes more difficult to establish that Ross was the individual captured in the specific camera footage. This creates reasonable doubt about whether Ross committed the particular theft being charged. The statement complicates the prosecution's narrative by suggesting a broader pattern involving more than one perpetrator."
},
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. On November 2nd, an incident of alleged credit card fraud was reported involving unauthorized purchases totaling over $2,000. The cardholder, Melissa Grant, stated that she noticed multiple charges from a local electronics store that she did not authorize. Store records indicate that the purchases were made in person using a physical card. A cashier, Andre Willis, recalled a customer matching a general description of a young adult male making several purchases but could not confirm identity. Surveillance footage shows an individual using a card at checkout, but the face is partially obscured by a hat. Another employee, Carla Ruiz, stated that the individual appeared nervous and rushed through the transaction. Police later identified a suspect, Brian Lee, who had previously been in possession of a card matching the same bank and design. Lee claims he found the card on the ground and did not use it. No fingerprints were recovered from the card reader.",
  "input": "How does Carla Ruiz's testimony support the prosecution's case theory?",
  "output": "Carla Ruiz's testimony supports the prosecution by describing behavior consistent with someone knowingly committing fraud. Her observation that the individual appeared nervous and rushed suggests consciousness of wrongdoing. This behavioral evidence strengthens the inference that the person using the card was aware it was not theirs. When combined with surveillance footage and the suspect's connection to a similar card, it helps build a circumstantial case of intentional unauthorized use."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A parking lot collision occurred outside a shopping center involving two vehicles, resulting in damage but no injuries. Driver A, Hannah Cole, claims she was reversing slowly when Driver B, Marcus Young, sped into the lot and struck her vehicle. Young contends that he was already driving through the lane when Cole suddenly backed out without checking. A pedestrian, Lily Evans, stated that she saw Cole's vehicle moving backward but could not determine the speed of either vehicle. Another witness, Derek Shaw, claimed that Young appeared to be driving faster than typical for a parking lot. Security camera footage captures the collision but does not clearly indicate vehicle speeds or right-of-way. No citations were issued at the scene.",
  "input": "How does Derek Shaw's testimony benefit the defense if representing Hannah Cole?",
  "output": "Derek Shaw's testimony benefits Hannah Cole's defense by suggesting that Marcus Young may have been driving at an unsafe speed for a parking lot. If Young was traveling too fast, it could reduce Cole's liability or shift fault toward Young. This supports the argument that even if Cole was reversing, Young's excessive speed contributed significantly to the collision. The testimony introduces doubt about Young's claim that he was driving normally and exercising proper caution."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A suspected case of academic cheating occurred during a final exam at a high school. The teacher, Mr. Daniels, reported observing a student, Olivia Park, frequently glancing at another student's paper during the test. Park denies cheating and claims she was simply thinking and looking around the room. Another student, Ethan Zhao, stated that he noticed Park turning her head repeatedly in the direction of a nearby student but could not confirm whether she was copying answers. A second teacher reviewing seating arrangements noted that the desks were placed relatively close together due to limited space. No physical evidence, such as notes or electronic devices, was found. The exam results showed that Park's answers were similar but not identical to the nearby student's responses.",
  "input": "What fact in this case packet weakens the claim of intentional cheating?",
  "output": "The fact that the desks were placed close together weakens the claim of intentional cheating because it creates the possibility of unintentional visibility. If students were seated in close proximity, it would be easier to glance in another direction without the intent to copy. Additionally, the answers being similar but not identical suggests independent work rather than direct duplication. These factors introduce reasonable doubt about whether any observed behavior constituted deliberate cheating."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A dog bite incident occurred in a neighborhood park involving a child and a pet dog. The child's parent, Laura Benson, claims that the dog, owned by Kevin Patel, was unleashed and approached aggressively before biting the child. Patel states that the dog is typically well-behaved and that the child may have startled or provoked it. A witness, Sandra Lopez, stated that she saw the dog running freely without a leash shortly before the incident. Another witness, Tom Riley, claimed that the child was running toward the dog and waving their arms. Park rules clearly require dogs to be leashed at all times. The dog has no prior record of aggressive behavior.",
  "input": "How does Tom Riley's testimony support the defense's case theory?",
  "output": "Tom Riley's testimony supports the defense by suggesting that the child may have contributed to the incident by approaching the dog in a sudden or provocative manner. If the child startled the dog, it could explain the dog's reaction without indicating inherent aggression. This helps shift some responsibility away from the owner and challenges the claim that the dog acted unprovoked. The testimony provides an alternative explanation for the behavior leading to the bite."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A dispute arose over alleged workplace harassment at a small marketing firm. The complainant, Jessica Moore, stated that her supervisor, Alan Reed, made repeated inappropriate comments over several weeks. Reed denies the allegations and claims that any remarks were taken out of context. A coworker, Natalie Singh, stated that she overheard some conversations but did not interpret them as inappropriate at the time. Another employee, Victor Chen, reported that Moore had previously expressed discomfort about Reed's comments in private. There are no recordings or written communications documenting the alleged remarks. Human resources records show no prior complaints against Reed.",
  "input": "How does the lack of documented evidence impact the prosecution's case?",
  "output": "The lack of documented evidence weakens the prosecution's case by making it more difficult to objectively verify the alleged harassment. Without recordings, written messages, or prior complaints, the case relies heavily on conflicting personal accounts. This creates challenges in establishing credibility and proving a pattern of behavior. The absence of corroborating documentation introduces reasonable doubt about the extent and nature of the alleged conduct."
},
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. On April 14th, an alleged case of identity theft was reported after several unauthorized bank withdrawals were made from an ATM over a two-day period. The account holder, Daniel Perez, stated that he had not shared his PIN with anyone and only noticed the missing funds after checking his account online. Bank surveillance footage shows an individual using the ATM multiple times, but the person’s face is partially obscured by a hood and sunglasses. A nearby shop owner, Irene Kwan, reported seeing someone loitering near the ATM late at night, appearing to watch others enter their PINs. Another witness, Malik Johnson, stated that he saw a person drop a card near the ATM but did not see who picked it up. Police later identified a suspect, Tyler Boone, who had prior access to the area and was found with multiple bank cards in his possession. Boone claims he found the cards scattered and did not use them.",
  "input": "How does Irene Kwan's testimony support the prosecution's case theory?",
  "output": "Irene Kwan's testimony supports the prosecution by suggesting that the suspect may have been engaging in suspicious behavior consistent with planning identity theft. Her observation of someone loitering and watching others enter their PINs indicates possible intent to gather sensitive information. This behavior aligns with the unauthorized ATM withdrawals that occurred shortly afterward. It strengthens the inference that the individual was not acting innocently but was deliberately attempting to exploit bank customers."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A confrontation occurred at a public basketball court resulting in minor injuries to one participant, Jason Miller. Miller claims that another player, Andre Scott, intentionally tripped him during a game after a disagreement over a foul. Scott denies this and claims that Miller fell accidentally while attempting a fast movement. A spectator, Chloe Rivera, stated that she saw the two arguing but did not clearly see any contact between them. Another witness, Ben Carter, claimed that Scott extended his leg in a way that could have caused the fall. There is no video footage of the incident. Miller suffered a sprained ankle but continued to play briefly afterward.",
  "input": "How does Chloe Rivera's testimony benefit the defense's case theory?",
  "output": "Chloe Rivera's testimony benefits the defense by highlighting uncertainty about whether any intentional contact occurred. Since she did not clearly observe Scott trip Miller, her account weakens claims of deliberate action. This lack of clear observation supports the argument that the fall could have been accidental. It introduces reasonable doubt about whether Scott intentionally caused the injury."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A case of alleged fare evasion occurred on a city bus where a passenger, Elena Torres, was accused of not paying the required fare. The bus driver, Robert Singh, stated that he did not see Torres tap her transit card when boarding. Torres claims that she did tap her card but that the scanner may not have registered it properly. Another passenger, David Lin, stated that he heard the usual beep sound when Torres boarded but was not watching directly. A second passenger, Maria Gomez, reported that the bus was crowded and that it was difficult to clearly see each person boarding. Transit system logs do not show a recorded fare for Torres at that time.",
  "input": "What fact supports the defense's claim that Elena Torres attempted to pay the fare?",
  "output": "David Lin's statement that he heard the characteristic beep sound when Torres boarded supports the defense's claim that she attempted to pay. This suggests that her card may have been scanned even if it was not properly recorded in the system logs. The crowded conditions described by another witness further explain how the driver might have missed the transaction. Together, these factors introduce doubt about whether Torres intentionally avoided paying."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A suspected case of property damage occurred when a parked car was found with a shattered side mirror on a residential street. The owner, Karen Blake, reported that the damage likely occurred overnight. A neighbor, Samuel Ortiz, stated that he heard a loud noise around midnight and saw a person walking quickly away from the area but could not identify them. Another resident, Lucy Park, reported that garbage collection trucks pass through the street late at night and could have accidentally caused the damage. No surveillance footage is available, and no physical evidence such as paint transfer was found on the vehicle. Police questioned a local resident, Eric Dunn, who was seen walking in the area but denied any involvement.",
  "input": "How does Lucy Park's statement affect the prosecution's case theory?",
  "output": "Lucy Park's statement weakens the prosecution's case by introducing an alternative explanation for the damage. If a garbage truck could have accidentally struck the vehicle, it reduces the likelihood that the damage was intentionally caused by a person. This creates reasonable doubt about whether a crime occurred at all. The lack of physical evidence further supports this uncertainty."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. An incident involving alleged trespassing occurred at a construction site that was closed to the public. The site manager, Paul Edwards, reported that tools were found moved and some materials were disturbed. A security guard, Nina Flores, stated that she saw an individual enter the site through a partially open gate late in the evening. The individual was wearing casual clothing and did not appear to be a worker. Another witness, Carlos Vega, claimed that the gate had been left open earlier in the day and that several people had walked near the entrance. Police later identified a suspect, Jordan Miles, who admitted to entering the site but claimed he believed it was open to the public and did not see any warning signs. No items were reported missing.",
  "input": "How does Carlos Vega's testimony support the defense's case theory?",
  "output": "Carlos Vega's testimony supports the defense by suggesting that the construction site may not have been clearly secured. If the gate was left open and people were seen walking near the entrance, it strengthens the argument that Jordan Miles could have reasonably believed the area was accessible. This undermines the claim that he knowingly trespassed. The lack of stolen items further supports the idea that there was no criminal intent."
},
    {
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. On January 9th, a suspected case of mail theft was reported in an apartment complex where several residents complained about missing letters and packages. One resident, Olivia Grant, stated that she saw an unfamiliar individual near the mailboxes around 3:00 PM, appearing to open multiple compartments. Another resident, Kevin Iyer, reported that he noticed his mailbox slightly ajar later that day but did not see anyone tampering with it. A third witness, Hannah Brooks, claimed that the mailbox area had been malfunctioning recently, with some doors not fully locking. Security camera footage shows a person near the mailboxes, but the angle does not clearly show whether they were accessing multiple units. Police later questioned a suspect, Ryan Cole, who was found with several pieces of mail addressed to different individuals. Cole claims he found the mail scattered on the ground and was attempting to return it.",
  "input": "How does Hannah Brooks' statement impact the prosecution's case theory?",
  "output": "Hannah Brooks' statement weakens the prosecution's case by suggesting that the mailboxes may not have been secure. If the compartments were not locking properly, it creates the possibility that mail could have fallen out or been easily accessed without intentional theft. This supports the defense's claim that Ryan Cole may have found the mail rather than stolen it. The statement introduces reasonable doubt about whether the suspect deliberately took the items."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A dispute arose at a grocery store involving an alleged case of food tampering. A customer, Mark Feldman, claimed that he observed another shopper, Lisa Tran, opening a sealed container and placing it back on the shelf. Tran denies this and states that she was only examining the product before deciding not to purchase it. A store employee, Jordan Wells, reported seeing Tran holding the container but did not witness any tampering. Another customer, Priya Nair, stated that she noticed the seal on a similar container appeared loose but could not confirm how it became that way. Store surveillance footage shows Tran handling the product but does not clearly show whether the seal was broken. No direct evidence of tampering was found.",
  "input": "What fact in this case packet creates doubt about whether Lisa Tran tampered with the product?",
  "output": "The lack of clear surveillance footage showing the seal being broken creates doubt about whether Lisa Tran tampered with the product. Although she was seen handling the container, no witness directly observed her opening it. Additionally, the report of other loose seals suggests that the issue may not be isolated to her actions. This absence of direct evidence weakens the claim of intentional tampering."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A minor fire broke out in a school science lab after hours, causing limited damage. The school administrator reported that all students had left the building, but a janitor, Luis Martinez, stated that he saw a student, Emily Park, near the lab shortly before the fire started. Park claims she returned briefly to retrieve a forgotten notebook and did not handle any equipment. Another staff member, Claire Donovan, reported that some lab equipment had been left plugged in earlier in the day. Fire investigators determined that the fire may have originated from an electrical source but could not rule out human involvement. No accelerants were found, and no clear evidence of intentional wrongdoing was identified.",
  "input": "How does Claire Donovan's statement affect the defense's case theory?",
  "output": "Claire Donovan's statement supports the defense by introducing a plausible accidental cause for the fire. If lab equipment was left plugged in, it could have caused an electrical malfunction leading to the fire. This reduces the likelihood that Emily Park's presence was responsible for the incident. The statement provides an alternative explanation that does not involve intentional or negligent actions by the accused."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. An incident involving alleged ticket scalping occurred outside a concert venue. Event staff reported that a man, Victor Alvarez, was selling tickets at a price significantly higher than face value. Alvarez claims he was simply reselling tickets he had legally purchased and was unaware of any restrictions. A witness, Daniel Cho, stated that he saw Alvarez actively negotiating prices with multiple buyers. Another witness, Sophia Reed, claimed that she saw other individuals also selling tickets in the same area. Venue policy prohibits unauthorized resale of tickets on-site. No undercover purchase was conducted to confirm the transaction details.",
  "input": "How does Sophia Reed's testimony impact the prosecution's case theory?",
  "output": "Sophia Reed's testimony weakens the prosecution's case by indicating that multiple individuals were engaged in similar behavior. This makes it harder to single out Victor Alvarez as uniquely responsible for violating the policy. It also suggests that enforcement may not have been consistent, which could affect how the situation is interpreted. The presence of others doing the same activity introduces ambiguity about intent and selective enforcement."
},
{
  "instruction": "You are a Senior Proescutor/Defender. Based ONLY on the provided statement, list the 3 strongest pieces of physical evidence (DNA, Serial Numbers, etc.) and explain why they prove guilt Beyond a Reasonable Doubt. DO NOT mention evidence not found in the text.. A reported case of noise disturbance occurred at a residential building where several tenants complained about loud music late at night. One tenant, Rachel Kim, stated that the noise was coming from the apartment of Jason Wu and continued past midnight. Wu denies playing loud music and claims he was watching television at a normal volume. Another resident, Alex Gomez, reported hearing music but was unsure which apartment it originated from. A third tenant, Linda Park, stated that sound often travels between units in the building, making it difficult to pinpoint the exact source. No recordings or direct measurements of noise levels were taken.",
  "input": "How does Linda Park's statement support the defense's case theory?",
  "output": "Linda Park's statement supports the defense by highlighting the difficulty of accurately identifying the source of the noise. If sound travels easily between units, it raises the possibility that the music came from a different apartment. This undermines the certainty of the complaint against Jason Wu. The lack of direct evidence further strengthens the argument that the claim cannot be definitively proven."
},
]


with open('training_data.json', 'w') as f:
    json.dump(data, f)

print(f"✅ Saved {len(data)} training examples.")

In [ ]:
from datasets import Dataset
import json

HF_USERNAME = "hobbesthecomputerscientist"

with open('training_data.json', 'r') as f:
    data = json.load(f)

# Format each flashcard into the instruction/input/output template
# This specific format (### Instruction / ### Input / ### Response)
# is called the Alpaca format — Phi-2 was trained to understand it
formatted = []
for example in data:
    formatted.append({
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    })

dataset = Dataset.from_list(formatted)
dataset.push_to_hub(f"{HF_USERNAME}/mock-trial-data")

print(f"Dataset uploaded to huggingface.co/datasets/{HF_USERNAME}/mock-trial-data")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "trl", "transformers", "peft", "-q", "--upgrade"])

import torch
import gc
torch.cuda.empty_cache()
gc.collect()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

HF_USERNAME = "hobbesthecomputerscientist"

print("Step 1/4: Loading base model...")

model_name = "microsoft/phi-1_5"  # ← smaller than phi-2, fits in T4 easily
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
config.pad_token_id = tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Step 2/4: Setting up LoRA...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("Step 3/4: Loading and tokenizing your training data...")
dataset = load_dataset(f"{HF_USERNAME}/mock-trial-data")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,        # ← reduced from 512 to save memory
        padding="max_length"
    )

tokenized_dataset = dataset["train"].map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

print("Step 4/4: Starting training! This takes ~10-20 minutes...")

training_args = TrainingArguments(
    output_dir="./mock-trial-model",
    num_train_epochs=5,
    per_device_train_batch_size=1,   # ← reduced from 4 to save memory
    gradient_accumulation_steps=4,   # ← simulates batch of 4 without the memory cost
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit"         # ← memory efficient optimizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()
print("Training complete! Move to Cell 6.")

In [ ]:
HF_USERNAME = "hobbesthecomputerscientist" # Double check this is your username
REPO_NAME = "mock-trial-ai-v2"

print(f"🚀 Attempting to push to {HF_USERNAME}/{REPO_NAME}...")

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print("✅ Push complete!")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, PhiConfig
import gradio as gr
import fitz  # PyMuPDF
import torch

# --- CONFIGURATION ---
HF_USERNAME = "hobbesthecomputerscientist"
MODEL_ID = f"{HF_USERNAME}/mock-trial-ai-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
config = PhiConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.model_type = "phi"
if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
    config.pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id else 50256

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, config=config, trust_remote_code=True, dtype=torch.float16, device_map="auto")
tokenizer.pad_token = tokenizer.eos_token
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# --- UTILS ---

def clean_ai_response(text):
    """Cuts off hallucinations while allowing for longer legal explanations."""
    stop_words = ["Illustration", "Example", "Exercise", "### Instruction", "Scenario", "Note:", "Discussion"]
    for word in stop_words:
        if word in text: text = text.split(word)[0]
    return text.strip()

def extract_pdf_text(pdf_file):
    if pdf_file is None: return ""
    text = ""
    path = pdf_file.name if hasattr(pdf_file, 'name') else pdf_file
    with fitz.open(path) as doc:
        for page in doc: text += page.get_text()
    return text[:4500]

# --- MODE LOGIC ---

def mock_trial_engine(pdf, mode, name, exam_type, user_input):
    case_text = extract_pdf_text(pdf) if pdf else "No PDF provided."

    if mode == "🎭 Witness Simulator":
        # Force the AI to stay in persona and be defensive
        instruction = (
            f"You are the witness {name}. ROLE: {exam_type}. "
            f"STRICT RULES: Use ONLY the provided case facts. If asked a trick question or a lie, "
            f"deny it firmly based on the text. Do not agree with the attorney. "
            f"TEXT: {case_text}"
        )
        temp, rep_pen = 0.2, 1.2

    elif mode == "⚖️ Objection Checker":
        # Standalone Judge logic
        instruction = (
            "You are a HIGH COURT JUDGE. Your only job is to rule on the input question. "
            "Is it Leading? Hearsay? Speculation? Relevance? Answer with: "
            "'RULING: [Objection Type] - REASONING: [Legal Explanation]' "
            "DO NOT answer the question as a witness."
        )
        temp, rep_pen = 0.01, 1.1

    elif mode == "🎯 Objection Practice":
        # Generate a question to test the user
        instruction = (
            f"Generate a realistic courtroom question for the witness {name}. "
            "The question should be intentionally flawed (Leading or Hearsay) to test a student. "
            f"CASE CONTEXT: {case_text}"
        )
        temp, rep_pen = 0.7, 1.0

    else: # Case Analysis
        instruction = f"Provide a detailed strategic analysis of the case packet: {case_text}"
        temp, rep_pen = 0.3, 1.1

    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{user_input}\n\n### Response:"
    res = generator(prompt, max_new_tokens=300, do_sample=(temp > 0), temperature=max(temp, 0.01), repetition_penalty=rep_pen)
    return clean_ai_response(res[0]['generated_text'].split("### Response:")[-1])

def solve_practice(pdf, question, guess):
    case_text = extract_pdf_text(pdf)
    # The 'Coach' prompt: compares guess to the actual question
    instruction = (
        f"The question asked was: '{question}'. The student's objection was: '{guess}'. "
        "Explain if the student is correct. Define the legal rule (Leading/Hearsay/etc.) "
        f"and cite if the question violates it based on: {case_text}"
    )
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\nEvaluate the ruling.\n\n### Response:"
    res = generator(prompt, max_new_tokens=250, do_sample=False, repetition_penalty=1.2)
    return clean_ai_response(res[0]['generated_text'].split("### Response:")[-1])

# --- UI (Improved Layout) ---

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏛️ Mock Trial AI")

    with gr.Tab("🎯 Objection Practice"):
        op_pdf = gr.File(label="Upload Case Packet")
        op_name = gr.Textbox(label="Witness to Question")
        gen_btn = gr.Button("Generate Attorney Question")
        generated_q = gr.Textbox(label="AI Attorney Question:", lines=2)

        user_choice = gr.Radio(["No Objection", "Leading", "Hearsay", "Speculation", "Relevance", "Argumentative"], label="Your Objection Choice")
        solve_btn = gr.Button("Submit Ruling", variant="primary")
        explanation = gr.Textbox(label="Judge's Feedback", lines=6)

        gen_btn.click(fn=lambda p,n: mock_trial_engine(p,"🎯 Objection Practice",n,"","Give me a question."), inputs=[op_pdf, op_name], outputs=[generated_q])
        solve_btn.click(solve_practice, inputs=[op_pdf, generated_q, user_choice], outputs=explanation)

    with gr.Tab("⚖️ Objection Checker"):
        gr.Markdown("Input a question here to see if it is objectionable. No PDF required.")
        oc_input = gr.Textbox(label="Question to Rule On", placeholder="e.g. You saw him do it, didn't you?")
        oc_btn = gr.Button("Judge's Ruling", variant="primary")
        oc_output = gr.Textbox(label="Rulings & Reasoning", lines=5)
        oc_btn.click(fn=lambda i: mock_trial_engine(None,"⚖️ Objection Checker","","",i), inputs=oc_input, outputs=oc_output)

    with gr.Tab("🎭 Witness Simulator"):
        with gr.Row():
            ws_pdf = gr.File(label="Upload PDF")
            with gr.Column():
                ws_name = gr.Textbox(label="Witness")
                ws_type = gr.Radio(["Direct", "Cross"], label="Mode", value="Cross")
        ws_input = gr.Textbox(label="Attorney Question")
        ws_btn = gr.Button("Question Witness")
        ws_output = gr.Textbox(label="Response")
        ws_btn.click(fn=lambda p,n,t,i: mock_trial_engine(p,"🎭 Witness Simulator",n,t,i), inputs=[ws_pdf, ws_name, ws_type, ws_input], outputs=ws_output)

    with gr.Tab("📋 Case Analysis"):
        ca_pdf = gr.File(label="Upload PDF")
        ca_input = gr.Textbox(label="Analysis Query")
        ca_btn = gr.Button("Execute Analysis")
        ca_output = gr.Textbox(label="Strategic Breakdown")
        ca_btn.click(fn=lambda p,i: mock_trial_engine(p,"📋 Case Analysis","","",i), inputs=[ca_pdf, ca_input], outputs=ca_output)

demo.launch(share=True)